# PVGIS-only ST-GNN + SDE-Net multi-horizon pipeline

Reproducible orchestrator for the final t+1h, t+6h and t+12h runs on server **newzealand**.

Does **not** duplicate runner/analysis logic — it builds commands and reads the
CSVs they write, via `physiq_pv.experiments.sde_pipeline`.

Safety switches: `RUN_TRAINING`, `RUN_ANALYSIS`, `LOG_TO_WANDB`.

Each horizon trains an independent copy of the same paper-faithful model (Gaussian NLL with beta=0), changing only the target offset. The results section generates the existing figures for every horizon and additional direct comparisons.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)

from physiq_pv.experiments import sde_pipeline as pipe
print('repo root:', REPO_ROOT)

In [ ]:
PVGIS_DIR = pipe.PVGIS_DIR
EXPERIMENT = 'paper_faithful_gaussian'
TRAIN_NORMAL_ONLY = False  # True rimuove gli eventi rari da train/validation
ANOMALY_SOURCE = 'detector'  # 'climatology' | 'detector'
DETECTOR = 'mtgflow'  # 'mtgflow' | 'catch' | 'm2ad'
DETECTOR_SEED = 15
DETECTOR_ROOTS = {
    'mtgflow': Path('outputs/pvgis_mtgflow/downstream_dense') / f'seed_{DETECTOR_SEED}',
    'catch': Path('outputs/pvgis_catch_2005_2019'),
    'm2ad': Path('outputs/pvgis_m2ad_2005_2019'),
}
DETECTOR_ROOT = DETECTOR_ROOTS[DETECTOR]
if ANOMALY_SOURCE == 'detector':
    TEST_ANOMALY_SCORES = str(DETECTOR_ROOT / 'anomaly_scores.csv')
    TRAIN_ANOMALY_SCORES = str(DETECTOR_ROOT / 'train_anomaly_scores.csv')
else:
    TEST_ANOMALY_SCORES = pipe.TEST_ANOMALY_SCORES
    TRAIN_ANOMALY_SCORES = pipe.TRAIN_ANOMALY_SCORES
ANALYSIS_SCRIPT = pipe.ANALYSIS_SCRIPT

NEEDS_TRAIN_ANOMALIES = bool(TRAIN_NORMAL_ONLY) or ANOMALY_SOURCE == 'detector'
checks = {
    'PVGIS dir': Path(PVGIS_DIR).is_dir(),
    'test anomaly scores': Path(TEST_ANOMALY_SCORES).exists(),
    'train anomaly scores': (not NEEDS_TRAIN_ANOMALIES) or Path(TRAIN_ANOMALY_SCORES).exists(),
    'analysis script': Path(ANALYSIS_SCRIPT).exists(),
}
try:
    import physiq_pv.experiments.pvgis_stgnn_runner  # noqa: F401
    checks['runner importable'] = True
except Exception as e:
    checks['runner importable'] = False
    print('runner import error:', e)
for k, v in checks.items():
    print(('OK     ' if v else 'MISSING') + '  ' + k)

If the anomaly scores are **missing**, regenerate them (run in a terminal — not launched automatically):

In [ ]:
if not Path(TEST_ANOMALY_SCORES).exists() or (NEEDS_TRAIN_ANOMALIES and not Path(TRAIN_ANOMALY_SCORES).exists()):
    if ANOMALY_SOURCE == 'detector':
        detector_notebook = {'mtgflow': 'mtgflow_pvgis_workflow.ipynb', 'catch': 'pvgis_catch_pipeline.ipynb', 'm2ad': 'pvgis_m2ad_pipeline.ipynb'}[DETECTOR]
        print(f'Mancano i CSV {DETECTOR}: eseguire prima notebooks/{detector_notebook}')
    else:
        base = ('PYTHONPATH=$PWD python scripts/run_pvgis_climatology_anomaly_years.py'
            ' --pvgis-dir ' + PVGIS_DIR +
            ' --climatology-start-year 2005 --climatology-end-year 2018'
            ' --rolling-past-climatology'
            ' --quantile 0.975 --climatology-window-days 15 --min-climatology-years 3'
            ' --variables solar_irradiance_poa pv_power_output temperature_2m wind_speed_10m'
            ' --out-root outputs')
        print('# Test year 2019:')
        print(base + ' --years 2019')
        print()
        print('# Train years 2016-2018 (aggregated):')
        print(base + ' --years 2016,2017,2018 --aggregate-out-dir ' + str(Path(TRAIN_ANOMALY_SCORES).parent))
else:
    print('anomaly scores present.')

## 2. Multi-horizon configuration

In [ ]:
# === EXPERIMENT SELECTOR ===
# Three independent paper-faithful runs. Architecture, training data and
# hyperparameters stay fixed; only the forecast target offset changes.
FORECAST_HORIZONS = pipe.FORECAST_HORIZONS
SELECTED_HORIZON = 1  # backward-compatible alias for optional diagnostics
BASE_EXPERIMENT_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': f'paper_faithful_gaussian_{ANOMALY_SOURCE}_{DETECTOR}_ep60',
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': TRAIN_NORMAL_ONLY,
    'n_sde_steps': 4, 'sigma_max': 0.5,
    'sde_sigma_initial': 0.01, 'sde_sigma_warmup_epochs': 30,
    'ood_noise_std': 2.0, 'mc_samples': 10, 'seed': 1,
    'ood_smoke_test': True, 'ood_smoke_max_samples': 2048,
               'anomaly_source': ANOMALY_SOURCE,
               'detector_regional_quantile': 0.975,
}

HORIZON_CONFIGS = {
    horizon: {**BASE_EXPERIMENT_CONFIG, 'horizon': horizon}
    for horizon in FORECAST_HORIZONS
}
OUT_DIRS = {horizon: pipe.make_out_dir(config) for horizon, config in HORIZON_CONFIGS.items()}
RUN_NAMES = {horizon: pipe.make_run_name(config) for horizon, config in HORIZON_CONFIGS.items()}
BASE_CONFIG = HORIZON_CONFIGS[SELECTED_HORIZON]
out_dir = OUT_DIRS[SELECTED_HORIZON]
run_name = RUN_NAMES[SELECTED_HORIZON]
COMPARISON_OUT_DIR = Path(out_dir).parent / f'{Path(out_dir).name}_horizons_1_6_12'
print('experiment:', EXPERIMENT)
for horizon in FORECAST_HORIZONS:
    print(f't+{horizon:02d}h  {OUT_DIRS[horizon]}  ({RUN_NAMES[horizon]})')
print('comparison:', COMPARISON_OUT_DIR)

## 3. Training command

In [ ]:
TRAIN_COMMANDS = {
    horizon: pipe.build_train_command(
        HORIZON_CONFIGS[horizon], out_dir=OUT_DIRS[horizon],
        run_name=RUN_NAMES[horizon], pvgis_dir=PVGIS_DIR,
        test_anomaly_scores=TEST_ANOMALY_SCORES,
        train_anomaly_scores=TRAIN_ANOMALY_SCORES, device='cuda', use_wandb=True,
    )
    for horizon in FORECAST_HORIZONS
}
for horizon, command in TRAIN_COMMANDS.items():
    assert command[command.index('--horizon') + 1] == str(horizon)
    assert '--wandb' in command and '--no-wandb-upload-artifacts' in command
    print(f'\n# t+{horizon}h')
    print(command)
train_cmd = TRAIN_COMMANDS[SELECTED_HORIZON]  # legacy alias
print(' \\\n  '.join(train_cmd))

## 4. Run training

Set `RUN_TRAINING = True` to actually launch.

In [ ]:
RUN_TRAINING = True
EVAL_ONLY_PREDICTIONS_BY_HORIZON = {}  # es. {6: '/path/predictions.csv'}
REUSE_COMPLETED_RUNS = True
ALLOW_OVERWRITE = False

if RUN_TRAINING and EVAL_ONLY_PREDICTIONS_BY_HORIZON:
    raise ValueError('Scegliere training oppure evaluation-only, non entrambi.')
def _run_is_complete(path):
    root = Path(path)
    return all((root / name).is_file() for name in ('best_model.pt', 'predictions.csv', 'metrics_global.csv'))

if RUN_TRAINING:
    pending = [
        horizon for horizon in FORECAST_HORIZONS
        if not (REUSE_COMPLETED_RUNS and _run_is_complete(OUT_DIRS[horizon]))
    ]
    # Preflight all paths before launching the first expensive run.
    for horizon in pending:
        pipe.ensure_output_dir_available(OUT_DIRS[horizon], allow_overwrite=ALLOW_OVERWRITE)
    for horizon in FORECAST_HORIZONS:
        if horizon not in pending:
            print(f'[reuse] t+{horizon}h: {OUT_DIRS[horizon]}')
            continue
        print(f'[train] t+{horizon}h')
        subprocess.run(TRAIN_COMMANDS[horizon], check=True)
elif EVAL_ONLY_PREDICTIONS_BY_HORIZON:
    for horizon, source_predictions in EVAL_ONLY_PREDICTIONS_BY_HORIZON.items():
        config = HORIZON_CONFIGS[int(horizon)]
        eval_paths = pipe.relabel_detector_predictions_file(
            source_predictions, TEST_ANOMALY_SCORES, TRAIN_ANOMALY_SCORES,
            OUT_DIRS[int(horizon)],
            regional_quantile=config['detector_regional_quantile'],
            min_temporal_coverage=config['detector_min_temporal_coverage'],
            allow_overwrite=ALLOW_OVERWRITE,
        )
        print(f'Evaluation-only t+{horizon}h:', eval_paths)
else:
    print('Training ed evaluation-only disattivati.')

## 5. Post-hoc analysis

In [ ]:
RUN_ANALYSIS = True

ANALYSIS_COMMANDS = {
    horizon: pipe.build_analysis_command(OUT_DIRS[horizon], HORIZON_CONFIGS[horizon])
    for horizon in FORECAST_HORIZONS
}
analysis_cmd = ANALYSIS_COMMANDS[SELECTED_HORIZON]  # legacy alias
print(' \\\n  '.join(analysis_cmd))
if RUN_ANALYSIS:
    for horizon, command in ANALYSIS_COMMANDS.items():
        predictions_path = Path(OUT_DIRS[horizon]) / 'predictions.csv'
        if not predictions_path.is_file():
            raise FileNotFoundError(f'Mancano le predizioni t+{horizon}h: {predictions_path}')
        print(f'[analysis] t+{horizon}h')
        subprocess.run(command, check=True)
else:
    print('RUN_ANALYSIS is False — not launching.')

## 6. Results & figures

Reads the CSVs written for t+1h, t+6h and t+12h, then shows: summary tables, the
SDE band (predictive std) by anomaly group, and the same post-hoc figures for every horizon
(per-sample MAE/NMPIL boxplots + pooled PICP/CLC bar charts, one figure per
production-percentage bin, split by the pointwise MTGFlow label matched on
`(location, timestamp)`). Every post-hoc normal/rare plot requires `anomaly_group`;
the regional `event_group` is not accepted as a fallback. The ST-GNN SDE head
reports epistemic and aleatoric components separately. A final section directly compares predictions and metrics across horizons.

In [ ]:
def _read(run_dir, name):
    p = Path(run_dir) / name
    return pd.read_csv(p) if p.exists() else None

RESULT_FILES = ['metrics_global.csv','metrics_daytime.csv','metrics_by_anomaly_label.csv',
                'residual_bias_and_bin_metrics.csv','daytime_bin_summary.csv',
                'daytime_bin_anomaly_metrics.csv','frequency_weighted_bin_summary.csv',
                'uncertainty_response.csv','sharpness_overview.csv']
RESULTS_BY_HORIZON = {
    horizon: {name: _read(OUT_DIRS[horizon], name) for name in RESULT_FILES}
    for horizon in FORECAST_HORIZONS
}
results = RESULTS_BY_HORIZON[SELECTED_HORIZON]  # legacy alias
for horizon, horizon_results in RESULTS_BY_HORIZON.items():
    print(f'\n# t+{horizon}h')
    for name, frame in horizon_results.items():
        print((('OK  ' if frame is not None else '--  ') + name) + (('  ' + str(frame.shape)) if frame is not None else ''))

In [ ]:
for horizon, horizon_results in RESULTS_BY_HORIZON.items():
    print(f'\n## t+{horizon}h')
    sharp = horizon_results['sharpness_overview.csv']
    if sharp is not None:
        cols = [c for c in ['scope','count','picp','mae','rmse','mean_std','mpiw','nmpil'] if c in sharp.columns]
        display(sharp[cols])
    bins = horizon_results['daytime_bin_summary.csv']
    if bins is not None:
        display(bins)
    freq = horizon_results['frequency_weighted_bin_summary.csv']
    if freq is not None:
        display(freq)
    unc = horizon_results['uncertainty_response.csv']
    if unc is not None:
        display(unc)

In [ ]:
# SDE total predictive std by anomaly group. Seasonal anomaly labels are
# evaluation strata only.
for horizon in FORECAST_HORIZONS:
    _pred = Path(OUT_DIRS[horizon]) / 'predictions.csv'
    if not _pred.exists():
        print(f't+{horizon}h predictions.csv not found — run training first.')
        continue
    _df = pd.read_csv(_pred, usecols=['anomaly_group', 'solar_irradiance_poa_target',
                                      'y_pred_std'])
    _day = _df[_df['solar_irradiance_poa_target'] > 10.0]
    print(f't+{horizon}h')
    display(_day.groupby('anomaly_group')[['y_pred_std']].mean())

In [ ]:
import pandas as pd
# Epistemic vs aleatoric for every forecast horizon.
for horizon in FORECAST_HORIZONS:
    prediction_path = Path(OUT_DIRS[horizon]) / 'predictions.csv'
    _ea = pd.read_csv(
        prediction_path,
        usecols=['epistemic_std', 'aleatoric_std', 'solar_irradiance_poa_target'],
    )
    _ea_day = _ea[_ea['solar_irradiance_poa_target'] > 10]
    means = _ea_day[['epistemic_std', 'aleatoric_std']].mean()
    print(f't+{horizon}h:', means.to_dict())
    print('epistemic / aleatoric ratio:', round(means['epistemic_std'] / means['aleatoric_std'], 4))

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

# Predictions of the CURRENTLY SELECTED experiment — derive out_dir from
# BASE_CONFIG so a stale global out_dir can't point at the wrong run.
out_dir = pipe.make_out_dir(BASE_CONFIG)
PRED_CSV = str(Path(out_dir) / "predictions.csv")
print("reading:", PRED_CSV)
GAMMA, ETA = 0.95, 9.0
SOLAR_COL, SOLAR_THR = "solar_irradiance_poa_target", 10.0

# Primary intervals are exact quantiles of the sampled Gaussian mixture.
# A moment-matched Gaussian band is retained only as a diagnostic.
NLL_DIST = 'gaussian'
z = float(norm.ppf(0.5 + GAMMA / 2))

want = ["y_true", "y_pred_mean", "y_pred_std_raw", "lower_pi", "upper_pi",
        SOLAR_COL, "timestamp", "location"]
# Always load from the selected run's CSV (no stale in-memory df shortcut).
d = pd.read_csv(PRED_CSV, usecols=lambda c: c in want)

# --- daytime + finite ---
d = d[d[SOLAR_COL] > SOLAR_THR]
need = ["y_true", "y_pred_mean", "y_pred_std_raw"]
d = d[np.isfinite(d[need].to_numpy()).all(axis=1)].reset_index(drop=True)

y = d["y_true"].to_numpy(float)
mu = d["y_pred_mean"].to_numpy(float)
sig = d["y_pred_std_raw"].to_numpy(float)

rng = y.max() - y.min()                       # target_range (come report)
rmse = np.sqrt(np.mean((mu - y) ** 2))

clc = lambda picp, nmpil: nmpil * (1.0 + np.exp(-ETA * (picp - GAMMA)))  # = report CLC

def metrics(lo, hi, name):
    inside = (y >= lo) & (y <= hi)
    picp, mpiw = inside.mean(), np.mean(hi - lo)
    nmpil = mpiw / rng
    return dict(model=name, PICP=round(picp, 4), MPIW=round(mpiw, 2),
                NMPIL=round(nmpil, 4), MPIW_RMSE=round(mpiw / rmse, 3),
                CLC=round(clc(picp, nmpil), 4))

rows = []
# 1) modello, PI Gaussian-mixture primari salvati dal runner
rows.append(metrics(d["lower_pi"].to_numpy(float), d["upper_pi"].to_numpy(float), "model (saved PI)"))
# 2) diagnostica moment-matched Gaussian sulla varianza totale
rows.append(metrics(mu - z * sig, mu + z * sig, "model (Gaussian moment diagnostic)"))
# No test-fitted homoscedastic/climatology baselines: fitting them on y_test
# would leak evaluation targets and they are not part of this primary run.

res = pd.DataFrame(rows)[["model", "PICP", "MPIW", "NMPIL", "MPIW_RMSE", "CLC"]]
print(f"dist={NLL_DIST}" +
      f"  target_range={rng:.2f}  RMSE={rmse:.2f}  gaussian_z={z:.3f}  (gamma={GAMMA}, eta={ETA})")
print(res.to_string(index=False))
ood_smoke_path = Path(out_dir) / 'ood_smoke_metrics.csv'
if ood_smoke_path.exists():
    print('\nControlled pseudo-OOD smoke test (not real held-out OOD):')
    print(pd.read_csv(ood_smoke_path).to_string(index=False))

# --- reliability diagram della sola diagnostica Gaussian moment-matched ---
levels = np.array([0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99])
emp = np.array([((y >= mu - norm.ppf(.5 + L / 2) * sig) & (y <= mu + norm.ppf(.5 + L / 2) * sig)).mean() for L in levels])
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", label="ideale")
plt.plot(levels, emp, "o-", label="Gaussian moment diagnostic")
plt.xlabel("nominal coverage"); plt.ylabel("empirical coverage")
plt.title("Reliability diagram (daytime)"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(pd.DataFrame({"nominal": levels, "empirical": np.round(emp, 4)}).to_string(index=False))

In [ ]:
# Ogni orizzonte riceve gli stessi grafici post-hoc attualmente prodotti.
FIGURE_PATHS_BY_HORIZON = {
    horizon: pipe.build_posthoc_figures(OUT_DIRS[horizon])
    for horizon in FORECAST_HORIZONS
}
figure_paths = FIGURE_PATHS_BY_HORIZON[SELECTED_HORIZON]  # legacy alias

# Confronto diretto: se None, usa la prima località e una settimana attorno
# al suo massimo di produzione. Impostare date ISO per una finestra specifica.
COMPARISON_LOCATION = None
COMPARISON_START = None
COMPARISON_END = None
HORIZON_COMPARISON_FIGURES = pipe.build_horizon_comparison_figures(
    OUT_DIRS, COMPARISON_OUT_DIR, location=COMPARISON_LOCATION,
    start=COMPARISON_START, end=COMPARISON_END,
)
print('per-horizon figures:', {h: list(paths) for h, paths in FIGURE_PATHS_BY_HORIZON.items()})
print('comparison figures:', list(HORIZON_COMPARISON_FIGURES))

In [ ]:
from IPython.display import Image, display
for horizon, paths in FIGURE_PATHS_BY_HORIZON.items():
    print(f'\n## Existing figures — t+{horizon}h')
    for path in paths.values():
        display(Image(filename=str(path)))
print('\n## Direct 1h / 6h / 12h comparison')
for path in HORIZON_COMPARISON_FIGURES.values():
    display(Image(filename=str(path)))

## 7. W&B sweep (optional)

In [ ]:
print('Sweep disabled: this notebook defines three controlled horizon runs.')

In [ ]:
# Intentionally no W&B sweep registration in the multi-horizon notebook.

## 8. Log post-hoc results to W&B (optional)

Adds post-hoc scalars and figures to each matching W&B horizon run without uploading artifacts.

In [ ]:
LOG_TO_WANDB = True
if LOG_TO_WANDB:
    import wandb
    for horizon in FORECAST_HORIZONS:
        run = pipe.init_wandb_run_for_out_dir(
            wandb, OUT_DIRS[horizon], run_name=RUN_NAMES[horizon]
        )
        try:
            posthoc = pipe.log_posthoc_to_wandb(
                wandb, run, OUT_DIRS[horizon],
                figure_paths=(
                    FIGURE_PATHS_BY_HORIZON.get(horizon)
                    if 'FIGURE_PATHS_BY_HORIZON' in dir() else None
                ),
                upload_artifact=False,
            )
            print(f't+{horizon}h:', posthoc['summary'])
        finally:
            run.finish()
else:
    print('LOG_TO_WANDB is False. posthoc summary preview:')
    for horizon in FORECAST_HORIZONS:
        if Path(OUT_DIRS[horizon], 'sharpness_overview.csv').exists():
            print(f't+{horizon}h:', pipe.read_posthoc_summary(OUT_DIRS[horizon]))